mutable struct WaveParameters
    depth::Float64
    frequency::Float64
    wavenumber::Float64
    g::Float64
end

For this Julia struct, I want to create two constructors (internal or external): one that accepts depth, frequency and gravity, and another that accepts depth, wavenumber and gravity. For both, gravity should be optional and default to 9.81. For the first constructor, i want the wavenumber to be calculated from the input parameters, i doesn't matter what formula you use because i'll change it later. Similarly, for the second constructor, I want the frequency to be calculated inside the second constructor.

In [1]:
# mutable struct WaveParameters
#     depth::Float64
#     frequency::Float64
#     wavenumber::Float64
#     g::Float64
# end


# function WaveParameters(; depth::Real, frequency::Real, g::Real=9.81)
#     wavenumber = (frequency^2) / g 
#     return WaveParameters(Float64(depth), Float64(frequency), Float64(wavenumber), Float64(g))
# end


# function WaveParameters(; depth::Real, wavenumber::Real, g::Real=9.81)
#     frequency = sqrt(g * wavenumber)
#     return WaveParameters(Float64(depth), Float64(frequency), Float64(wavenumber), Float64(g))
# end;


# # Allow reading via aliases
# function Base.getproperty(obj::WaveParameters, sym::Symbol)
#     if sym === :h
#         return getfield(obj, :depth)
#     elseif sym === :ω
#         return getfield(obj, :frequency)
#     elseif sym === :k₀
#         return getfield(obj, :wavenumber)
#     else
#         return getfield(obj, sym)  # Default fieldname
#     end
# end

# # Allow writing via aliases
# function Base.setproperty!(obj::WaveParameters, sym::Symbol, val)
#     if sym === :h
#         return setfield!(obj, :depth, convert(Float64, val))
#     elseif sym === :ω
#         return setfield!(obj, :frequency, convert(Float64, val))
#     elseif sym === :k₀
#         return setfield!(obj, :wavenumber, convert(Float64, val))
#     else
#         return setfield!(obj, sym, convert(Float64, val))  # Default fieldname
#     end
# end
;



mutable struct WaveParameters
    depth::Float64
    frequency::Float64
    wavenumber::Float64
    g::Float64

    function WaveParameters(;
        depth::Real, frequency::Union{Real, Nothing}=nothing, wavenumber::Union{Real, Nothing}=nothing, g::Real=9.80665,
    )
        if !isnothing(frequency) && isnothing(wavenumber)
            k₀ = -1.0
            return new(Float64(depth), Float64(frequency), Float64(k₀), Float64(g))
        elseif !isnothing(wavenumber) && isnothing(frequency)
            ω = -2.0
            return new(Float64(depth), Float64(ω), Float64(wavenumber), Float64(g))
        else
            error("Either frequency or wavenumber must be given.")
        end
    end
end


function set_wave(;
    depth::Real, frequency::Union{Real, Nothing}=nothing, wavenumber::Union{Real, Nothing}=nothing, g::Real=9.80665,
)
    if !isnothing(frequency) && isnothing(wavenumber)
        k₀ = -1.0
        return WaveParameters(Float64(depth), Float64(frequency), Float64(k₀), Float64(g))
    elseif !isnothing(wavenumber) && isnothing(frequency)
        ω = -2.0
        return WaveParameters(Float64(depth), Float64(ω), Float64(wavenumber), Float64(g))
    else
        error("Either frequency or wavenumber must be given.")
    end
end;

In [2]:
# wave = WaveParameters()
# wave.depth

In [3]:
methods(WaveParameters)

# 3 methods for type constructor:
 [1] WaveParameters(; depth, frequency, wavenumber, g)
     @ In[1]:72
 [2] WaveParameters(depth::Float64, frequency::Float64, wavenumber::Float64, g::Float64)
     @ In[1]:51
 [3] WaveParameters(depth, frequency, wavenumber, g)
     @ In[1]:51

In [4]:
WaveParameters(depth=5.0, frequency=6.0)

WaveParameters(5.0, 6.0, -1.0, 9.80665)

In [5]:
WaveParameters(depth=5.0, frequency=6)

WaveParameters(5.0, 6.0, -1.0, 9.80665)

In [6]:
WaveParameters(depth=5.0, wavenumber=6.0)

WaveParameters(5.0, -2.0, 6.0, 9.80665)

In [7]:
WaveParameters(5.0, 6.0, 2.0, 9.0)

WaveParameters(5.0, 6.0, 2.0, 9.0)

In [6]:
WaveParameters(depth=5.0)

LoadError: Either frequency or wavenumber must be given.

In [7]:
WaveParameters(depth=5.0, frequency=6.0, wavenumber=2.0)

LoadError: Either frequency or wavenumber must be given.

In [8]:
@code_warntype WaveParameters(depth=5.0, frequency=6.0)

MethodInstance for Core.kwcall(::@NamedTuple{depth::Float64, frequency::Float64}, ::Type{WaveParameters})
  from kwcall(::NamedTuple, ::Type{WaveParameters}) @ Main In[1]:72
Arguments
  _::Core.Const(Core.kwcall)
  @_2::@NamedTuple{depth::Float64, frequency::Float64}
  @_3::Core.Const(WaveParameters)
Locals
  @_4::Union{Nothing, Float64}
  depth::Float64
  frequency::Float64
  wavenumber::Nothing
  g::Float64
Body::WaveParameters
1 ──       Core.NewvarNode(:(@_4))
│    %2  = Core.isdefined(@_2, :depth)::Core.Const(true)
└───       goto #6 if not %2
2 ── %4  = Core.getfield(@_2, :depth)::Float64
│    %5  = Main.Real::Core.Const(Real)
│    %6  = (%4 isa %5)::Core.Const(true)
└───       goto #4 if not %6
3 ──       goto #5
4 ──       Core.Const(:(Main.Real))
│          Core.Const(:(%new(Core.TypeError, Symbol("keyword argument"), :depth, %9, %4)))
└───       Core.Const(:(Core.throw(%10)))
5 ┄─       (@_4 = %4)
└───       goto #7
6 ──       Core.Const(:(Core.UndefKeywordError(:depth)))
└──

In [13]:
wave.ω

6.0

In [5]:
wave.depth

10.0

In [ ]:
wave

In [4]:
using Base: @kwdef

@kwdef mutable struct Person
    name::String
    age::Int
    country::String = "USA"  # This one has a default, so it's optional
end

# Usage - 'name' and 'age' MUST be provided as keywords
p1 = Person(; name="Alice", age=30)           # ✓ Works
p2 = Person(; name="Bob", age=25, country="Canada")  # ✓ Works
p3 = Person(name="Charlie", age=40)           # ✓ Also works (parentheses optional)
# p4 = Person(; age=30)                         # ✗ ERROR: missing required keyword 'name'
# p5 = Person("Alice", 30)                      # ✗ ERROR: positional arguments not allowed

Person("Charlie", 40, "USA")

In [5]:
p1

Person("Alice", 30, "USA")

In [1]:
struct StrictlyKeyword
    name::String
    id::Int

    # Inner constructor forcing keywords
    function StrictlyKeyword(; name::String, id::Int)
        new(name, id)
    end
end

In [4]:
t = StrictlyKeyword(name="asdf", id=12)

StrictlyKeyword("asdf", 12)